In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

In [2]:
import numpy as np
from loaders._load_vn30_binary import preprocess, VN30, TARGETS
from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import balanced_accuracy_score, confusion_matrix

In [3]:
# X_train, y_train = preprocess("ACB", verbose=True)["train"]

In [4]:
acc = []
for symbol in VN30:
    data = preprocess(symbol, lag=30)
    X_train, Y_train = data["train"]
    X_test, Y_test = data["test"]

    tscv = TimeSeriesSplit(n_splits=5)

    param_dist = {
        "C": [0.1, 0.3, 1, 3, 10, 30, 100],
        "gamma": ["scale", "auto", 0.01, 0.1, 1, 10],
        "kernel": ["rbf", "linear", "poly", "sigmoid"],
        "degree": [2, 3, 4]
    }

    search = RandomizedSearchCV(
        estimator=SVC(),
        param_distributions=param_dist,
        n_iter=10,
        cv=tscv,
        n_jobs=-1,
        random_state=42
    )

    search.fit(X_train, Y_train)

    train_preds = search.predict(X_train)
    test_preds = search.predict(X_test)

    print(f"Symbol: {symbol}")
    print(f"Train Balanced Accuracy: {balanced_accuracy_score(Y_train, train_preds)}")
    print(f"Test Balanced Accuracy: {balanced_accuracy_score(Y_test, test_preds)}")
    print(f"Test Confusion Matrix:\n{confusion_matrix(Y_test, test_preds)}\n")

    acc.append(balanced_accuracy_score(Y_test, test_preds))

# Mean of best 10
mean_acc = np.mean(sorted(acc)[-10:])
print(f"Mean Test Balanced Accuracy (Best 10): {mean_acc}")

Symbol: ACB
Train Balanced Accuracy: 0.6234304657444545
Test Balanced Accuracy: 0.48627450980392156
Test Confusion Matrix:
[[105  70]
 [ 96  57]]

Symbol: BCM
Train Balanced Accuracy: 0.5
Test Balanced Accuracy: 0.5
Test Confusion Matrix:
[[187   0]
 [141   0]]

Symbol: BID
Train Balanced Accuracy: 0.5711805555555556
Test Balanced Accuracy: 0.5
Test Confusion Matrix:
[[191   0]
 [137   0]]

Symbol: BVH
Train Balanced Accuracy: 0.5
Test Balanced Accuracy: 0.5
Test Confusion Matrix:
[[185   0]
 [143   0]]

Symbol: CTG
Train Balanced Accuracy: 0.38360213092023643
Test Balanced Accuracy: 0.5628354203935599
Test Confusion Matrix:
[[ 84  72]
 [ 71 101]]

Symbol: FPT
Train Balanced Accuracy: 1.0
Test Balanced Accuracy: 0.5516914145361169
Test Confusion Matrix:
[[87 72]
 [75 94]]

Symbol: GAS
Train Balanced Accuracy: 0.9532062391681109
Test Balanced Accuracy: 0.50003874917658
Test Confusion Matrix:
[[191   6]
 [127   4]]

Symbol: GVR
Train Balanced Accuracy: 0.37023957070672675
Test Balanced A

In [6]:
std_acc = np.std(sorted(acc)[-10:])
print(f"Standard Deviation of Test Balanced Accuracy: {std_acc}")

Standard Deviation of Test Balanced Accuracy: 0.02417365818282643
